# 2 · Inputs

Notebook 1 changed the model. This notebook changes what the model reads.

So far it has read four raw numbers per pixel, divided by 10,000 because we were told to. A person telling water from forest would not squint at raw numbers; they would use what is known about light: plants reflect near-infrared strongly, and water absorbs it. Let us give the model the same head start, train it, and check whether it helps, first in the scores and then on a map.

In [ ]:
from pathlib import Path

from huggingface_hub import snapshot_download

import geosave_engine as gs

root = Path(snapshot_download("fatmur/dynamicworld-300", repo_type="dataset", local_dir="data/dynamic_world"))

name = sorted((root / "val/images").iterdir())[0].name
image = gs.open_raster(root / "val/images" / name).isel(time=0)
label = gs.open_raster(root / "val/labels" / name).isel(time=0)

gs.stack({"image": image, "label": label}).gs.plot(cols=2)

## What are the four numbers?

Before improving the inputs we need to know what they are. The file says: each band records which light it measured. `B08` is near-infrared, the light plants reflect strongly.

In [ ]:
image.gs.spectral

## From stored numbers to reflectance

The files store reflectance, the share of light reflected, as whole numbers times 10,000. That scale is written in the file too, so `mask_and_scale()` can apply it instead of our hand-written `/ 10_000`. It also blanks pixels with no measurement as NaN, where dividing would have left a misleading 0.

The values now sit between 0 and 1:

In [ ]:
reflectance = image.gs.mask_and_scale()
reflectance.gs.statistics()

## Hints for vegetation and water

Plants absorb red and reflect near-infrared, so **NDVI** = (nir − red) / (nir + red) is high over vegetation and low over bare ground and water. **NDWI** = (green − nir) / (green + nir) does the same for water. `gs.features` has both; we name which band is which light.

Compare them with the label: each index already outlines a few classes the model has to learn.

In [ ]:
ndvi = gs.features.ndvi(reflectance, nir="B08", red="B04")
ndwi = gs.features.ndwi(reflectance, green="B03", nir="B08")

(label.gs.plot() + ndvi.gs.plot(cmap="RdYlGn") + ndwi.gs.plot(cmap="RdBu")).cols(3)

## The new inputs

`prepare` turns a tile into the model's inputs: reflectance plus the two indices, added as bands with `assign`. Bands are named, so trying another choice is a matter of names, such as `reflectance[["B04", "B03", "B02"]]` for RGB only.

In [ ]:
def prepare(image):
    reflectance = image.gs.mask_and_scale()
    ndvi = gs.features.ndvi(reflectance, nir="B08", red="B04")
    ndwi = gs.features.ndwi(reflectance, green="B03", nir="B08")
    return reflectance.assign(ndvi=ndvi, ndwi=ndwi)


inputs = prepare(image)
inputs.gs.variables

## Putting the bands on one footing

Reflectance sits between 0 and 1 while the indices run from -1 to 1, and near-infrared is brighter than blue almost everywhere. Networks learn faster when every input channel is centred on 0 with a similar spread, so the usual last step is to standardise: subtract each band's mean and divide by its standard deviation.

The statistics come from training tiles only, so validation stays unseen. Forty tiles are plenty to estimate six means and spreads.

In [ ]:
import pandas as pd
import torch

train_names = sorted(path.name for path in (root / "train/images").iterdir())
tiles = [prepare(gs.open_raster(root / "train/images" / name).isel(time=0)) for name in train_names[:40]]
pixels = torch.stack([tile.gs.to_tensor(dtype="float32") for tile in tiles]).nan_to_num(0)

# One value per band, shaped to broadcast over (bands, height, width).
mean = pixels.mean(dim=(0, 2, 3)).view(-1, 1, 1)
std = pixels.std(dim=(0, 2, 3)).view(-1, 1, 1)

pd.DataFrame({"mean": mean.flatten(), "std": std.flatten()}, index=inputs.gs.variables)

## Training on six bands

The dataset is notebook 0's with `prepare` in place of `/ 10_000`, then standardised. Two details for pixels the satellite did not measure, which are NaN:

- their label becomes 255, so the loss and metrics skip them as they skip unlabelled pixels;
- after standardising they become 0, the band's mean, a neutral number the model can read.

The model is notebook 0's too, with its first layer reading six channels instead of four.

In [ ]:
import lightning as L
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchmetrics import MetricCollection
from torchmetrics.classification import MulticlassAccuracy, MulticlassJaccardIndex


class Tiles(Dataset):
    def __init__(self, split):
        self.folder = root / split
        self.names = sorted(path.name for path in (self.folder / "images").iterdir())

    def __len__(self):
        return len(self.names)

    def __getitem__(self, index):
        name = self.names[index]
        image = prepare(gs.open_raster(self.folder / "images" / name).isel(time=0))
        label = gs.open_raster(self.folder / "labels" / name).isel(time=0)

        pixels = image.gs.to_tensor(dtype="float32")
        mask = label.gs.to_tensor(dtype="int64")[0]

        measured = ~pixels.isnan().any(dim=0)
        mask[~measured] = 255

        pixels = ((pixels - mean) / std).nan_to_num(0)
        return pixels, mask


class Segmenter(L.LightningModule):
    def __init__(self, bands):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(bands, 16, 3, stride=2, padding=1), nn.ReLU(),
            nn.Upsample(scale_factor=2), nn.Conv2d(16, 16, 3, padding=1), nn.ReLU(),
            nn.Conv2d(16, 10, 1),
        )
        metrics = MetricCollection({
            "miou": MulticlassJaccardIndex(num_classes=10, ignore_index=255),
            "accuracy": MulticlassAccuracy(num_classes=10, ignore_index=255, average="micro"),
        })
        self.val_metrics = metrics.clone(prefix="val_")

    def forward(self, image):
        return self.model(image)

    def training_step(self, batch, batch_index):
        image, label = batch
        loss = nn.functional.cross_entropy(self(image), label, ignore_index=255)
        self.log("train_loss", loss)
        return loss

    def validation_step(self, batch, batch_index):
        image, label = batch
        self.val_metrics(self(image), label)
        self.log_dict(self.val_metrics)

    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=1e-2)


train_loader = DataLoader(Tiles("train"), batch_size=8, shuffle=True)
val_loader = DataLoader(Tiles("val"), batch_size=8)

module = Segmenter(bands=len(inputs.gs.variables))
trainer = L.Trainer(max_epochs=3)
trainer.fit(module, train_loader, val_loader)

## Did it help?

This run logs beside notebooks 0 and 1 in `lightning_logs/`. Compare `val_miou` across the runs: the same small model, given reflectance and two indices instead of raw numbers.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir lightning_logs

Scores summarise every tile; a map shows where the model is right and wrong. Our tile is the first validation tile, so the dataset prepares it exactly as in training. Keep the best-scoring class at each pixel.

In [ ]:
pixels, _ = Tiles("val")[0]

module.eval()
with torch.no_grad():
    scores = module(pixels[None])[0]

classes = scores.argmax(0).numpy().astype("uint8")

`classes` is a bare array: no position, no class names. `gs.raster` puts it on the tile's grid, and `write_legend` names and colours the classes from the dataset's `classes.yaml`, so it plots beside the label with the same legend.

In [ ]:
import yaml

legend = yaml.safe_load((root / "classes.yaml").read_text())

predicted = gs.raster({"prediction": (("y", "x"), classes)}, image.gs.geobox)
predicted = predicted.gs.write_legend(legend["class_map"], legend["color_map"])

gs.stack({"label": label, "predicted": predicted}).gs.plot(cols=2)

To look at it in context, or hand it to someone using QGIS, save it as a Cloud-Optimized GeoTIFF. It keeps its position and class names, so reopening it lands it on a real map; zoom out to see where on Earth this tile is.

In [ ]:
files = predicted.gs.to_cog("predicted.tif", overwrite=True)
gs.open_raster(files).gs.explore()

## What happened

We replaced the hand-written `/ 10_000` with reflectance read from the file, added two indices that outline vegetation and water, standardised every band, kept unmeasured pixels out of the loss, trained the same small model on them, and compared the runs. The prediction went back on the tile's grid as a map that opens anywhere.

Our tiles stop us going further. They were chosen cloud-free, and real imagery is not. And the pretrained encoder from notebook 1 reads each pixel's date and location as well as its bands. Both mean fetching imagery ourselves.

| Next | Question |
| --- | --- |
| 3 · Your own data | How do I fetch imagery for my own labels, places and dates? |